# Module 4, Demo lab 1: Simulate data by study design & compare groups

Not graded · Follow along · pairs with [Demo lab 2 (tests & CIs)](module-4-lab-demo-tests) and [Project 4](../projects/project-4).

This demo is about *how data arise under different study designs* and *comparing groups with the right descriptive statistics and plots*, the Module 3 toolkit (`group_by`, `summarize`, `count`, boxplots, stacked bars). We do not run formal tests here; that is [Demo lab 2](module-4-lab-demo-tests).

| Situation | Design | Outcome | Compare with |
|---|---|---|---|
| 1 | RCT | numerical | group means/SD + boxplot |
| 2 | Observational + confounder | binary | crude vs subclassified rates + stacked bar |
| 3 | RCT | binary | proportions + 100% stacked bar |
| 4 | 3+ groups | numerical | group summaries + boxplots |
| 5 | Paired / matched | numerical | within-unit change |


## Code anatomy legend (demo notebooks only)

As you read through the demo and see R code, try to break it down into the following 4 components:

| Color | Meaning in code |
|-------|-----------------|
| <span style="color:#2563eb">Blue</span> | R functions and syntax (`rnorm`, `<-`, `()`, `~`) |
| <span style="color:#059669">Green</span> | Names you created (variables, data frames) |
| <span style="color:#dc2626">Red</span> | Values to change for your question or data |
| <span style="color:#7c3aed">Purple</span> | Important output to read carefully |


*Note:* Colab may highlight R syntax in its own colors. My anatomy colors appear only on my website, not in Colab.

In [ ]:
library(tidyverse)
# set.seed() makes the simulated "random" data reproducible:
# anyone who runs this notebook gets the same numbers.

## New functions in this demo: simulating data

These build the datasets, and you will reuse them in the templates and in Project 4.

- `set.seed(42)` fixes the random number stream so the same code gives the same numbers every run (useful for reproducible grading).
- `rnorm(n, mean, sd)` draws `n` values from a normal distribution; use it for a numerical outcome.
- `rbinom(n, 1, p)` draws `n` values that are 0 or 1, each 1 with probability `p`; use it for a binary outcome, then label 0/1 as no/yes.
- `sample(values, n, replace = TRUE, prob = ...)` draws `n` values from `values` with the given probabilities; use it to assign a group or a confounder level.
- `ifelse(condition, a, b)` returns `a` where the condition is TRUE and `b` where it is FALSE; use it to make one variable depend on another (for example, group membership depending on a confounder).
- `tibble(...)` collects the simulated columns into one data frame, one row per observational unit.

The describing tools (`group_by()`, `summarize()`, `count()`, and the `ggplot2` geoms) are from Module 3.

## Situation 1, RCT, numerical outcome

A sleep program (Treatment) vs Control; the response variable is nightly hours of sleep. In an RCT we simulate by drawing each group from its own population mean. Compare with group means/SD and a boxplot.

In [ ]:
set.seed(42)
sleep_rct <- tibble(
  group = factor(rep(c("Control", "Treatment"), each = 30),
                 levels = c("Control", "Treatment")),
  sleep = c(rnorm(30, mean = 6.8, sd = 1.0),
            rnorm(30, mean = 7.5, sd = 1.0))
)

sleep_rct |>
  group_by(group) |>
  summarize(n = n(), mean = mean(sleep), sd = sd(sleep),
            median = median(sleep), iqr = IQR(sleep))

ggplot(sleep_rct, aes(x = group, y = sleep)) +
  geom_boxplot() +
  labs(title = "Nightly sleep by assigned group (simulated RCT)",
       x = NULL, y = "Hours of sleep")

## Situation 2, Observational study with a confounder

Students choose a program (New vs Standard); the response variable is pass/fail. A confounder, prior preparation, makes well-prepared students both more likely to choose New *and* more likely to pass. Watch how the crude comparison differs from the subclassified one (within levels of prior prep).

In [ ]:
set.seed(7)
N <- 400
prior <- sample(c("High", "Low"), N, replace = TRUE, prob = c(0.5, 0.5))
# Observational: High-prep students are more likely to pick the New program
p_new <- ifelse(prior == "High", 0.70, 0.30)
program <- ifelse(rbinom(N, 1, p_new) == 1, "New", "Standard")
# Passing depends mostly on prior prep, only slightly on program
p_pass <- 0.55 + 0.25 * (prior == "High") + 0.02 * (program == "New")
passed <- rbinom(N, 1, pmin(p_pass, 0.98))

obs <- tibble(
  prior   = factor(prior, levels = c("Low", "High")),
  program = factor(program, levels = c("Standard", "New")),
  passed  = passed
)

# Crude comparison (ignores the confounder)
obs |>
  group_by(program) |>
  summarize(n = n(), pass_rate = mean(passed))

# Subclassified comparison (within levels of prior prep)
obs |>
  group_by(prior, program) |>
  summarize(n = n(), pass_rate = mean(passed), .groups = "drop")

# The confounder is distributed unevenly across programs -- that is the confounding
obs |>
  count(program, prior) |>
  ggplot(aes(x = program, y = n, fill = prior)) +
  geom_col(position = "fill", color = "white") +
  labs(title = "Prior-prep mix differs by program (the confounding)",
       x = NULL, y = "Proportion", fill = "Prior prep")

What to notice. The crude New-vs-Standard pass rates can look different mostly because New attracts more high-prep students. Once you compare *within* prior-prep levels, the program difference shrinks. This is exactly the kind of confounding Read 1 §1 warns about, and why a small p-value (Demo 2) does not by itself prove causation.

## Situation 3, RCT, binary outcome

A drug vs placebo; the response variable is recovered (yes/no). Compare with proportions and a 100% stacked bar.

In [ ]:
set.seed(42)
drug <- tibble(
  group = factor(rep(c("Placebo", "Drug"), each = 80),
                 levels = c("Placebo", "Drug")),
  recovered = factor(c(rbinom(80, 1, 0.45), rbinom(80, 1, 0.65)),
                     labels = c("No", "Yes"))
)

drug |>
  group_by(group) |>
  summarize(n = n(),
            recovered = sum(recovered == "Yes"),
            prop = mean(recovered == "Yes"))

ggplot(drug, aes(x = group, fill = recovered)) +
  geom_bar(position = "fill") +
  labs(title = "Recovery by group (simulated RCT)",
       x = NULL, y = "Proportion", fill = "Recovered?")

## Situation 4, More than two groups, numerical outcome

Three clinics; the response variable is blood-pressure drop. With 3+ groups we still describe the same way: group summaries and side-by-side boxplots. (The matching test is one-way ANOVA, see Demo 2.)

In [ ]:
set.seed(101)
three <- tibble(
  clinic  = factor(rep(c("A", "B", "C"), each = 40)),
  bp_drop = c(rnorm(40, 8, 3), rnorm(40, 10, 3), rnorm(40, 13, 3))
)

three |>
  group_by(clinic) |>
  summarize(n = n(), mean = mean(bp_drop), sd = sd(bp_drop),
            median = median(bp_drop), iqr = IQR(bp_drop))

ggplot(three, aes(x = clinic, y = bp_drop)) +
  geom_boxplot() +
  labs(title = "Blood-pressure drop by clinic (3 groups, simulated)",
       x = "Clinic", y = "BP drop")

## Situation 5, Paired / matched design

The same people are measured before and after. Because the two numbers come from one person, we summarize the within-person change (after − before), not two independent groups. (The matching test is the paired *t*-test, see Demo 2.)

In [ ]:
set.seed(202)
n <- 25
before <- rnorm(n, mean = 150, sd = 12)
after  <- before - rnorm(n, mean = 6, sd = 4)   # each person tends to drop
paired <- tibble(id = 1:n, before = before, after = after,
                 change = after - before)

paired |>
  summarize(n = n(), mean_change = mean(change), sd_change = sd(change),
            median_change = median(change))

ggplot(paired, aes(y = change)) +
  geom_boxplot() +
  labs(title = "Within-person change (after - before), simulated", y = "Change")

## Copy-and-adapt templates: the six cases

The situations above tell specific stories. The templates below are stripped down to generic names you can rename for your own project: `your_explanatory_variable` (the two groups you compare), `your_numerical_outcome` or `your_binary_outcome` (what you measure), and `your_confounder` (needed only for an observational design).

A group difference has three possible explanations (Read 1, Section 1), and your response is either numerical or binary, so there are six templates:

| Explanation | Numerical outcome | Binary outcome |
|---|---|---|
| By chance (RCT, no true effect) | Case 1 | Case 4 |
| Causation (RCT, true effect) | Case 2 | Case 5 |
| Confounding (observational) | Case 3 | Case 6 |

How to use one: pick the case whose explanation and outcome type match the story you want, rename the `your_*` columns to your real variables, and change the sample sizes, the means and SDs (numerical) or the probabilities (binary). In the by chance cases both groups come from the same population. In the causation cases the treatment group differs by a real effect and assignment is random, so there is no confounder. In the confounding cases an observational confounder drives both the group and the outcome, so you compare within confounder levels.

### Case 1, Numerical outcome, by chance (RCT, no true effect)

In [ ]:
set.seed(1)
n_per_group <- 40

# Random assignment, and BOTH groups come from the same population:
# any gap between the group means is chance alone.
chance_numeric <- tibble(
  your_explanatory_variable = factor(rep(c("control", "treatment"), each = n_per_group),
                                     levels = c("control", "treatment")),
  your_numerical_outcome    = c(rnorm(n_per_group, mean = 50, sd = 10),
                                rnorm(n_per_group, mean = 50, sd = 10))
)

chance_numeric |>
  group_by(your_explanatory_variable) |>
  summarize(n = n(),
            mean = mean(your_numerical_outcome),
            sd   = sd(your_numerical_outcome))

### Case 2, Numerical outcome, causation (RCT, true effect)

In [ ]:
set.seed(2)
n_per_group <- 40

# Random assignment, plus a REAL effect: the treatment mean is truly higher.
# Because assignment is random, there is no confounder to worry about.
cause_numeric <- tibble(
  your_explanatory_variable = factor(rep(c("control", "treatment"), each = n_per_group),
                                     levels = c("control", "treatment")),
  your_numerical_outcome    = c(rnorm(n_per_group, mean = 50, sd = 10),
                                rnorm(n_per_group, mean = 58, sd = 10))
)

cause_numeric |>
  group_by(your_explanatory_variable) |>
  summarize(n = n(),
            mean = mean(your_numerical_outcome),
            sd   = sd(your_numerical_outcome))

### Case 3, Numerical outcome, confounding (observational)

In [ ]:
set.seed(3)
N <- 200

# Observational: a confounder drives BOTH group membership AND the outcome,
# and the group itself has no true effect.
your_confounder <- sample(c("low", "high"), N, replace = TRUE, prob = c(0.5, 0.5))

# high-confounder units are more likely to land in the treatment group
p_treat <- ifelse(your_confounder == "high", 0.75, 0.25)
your_explanatory_variable <- ifelse(rbinom(N, 1, p_treat) == 1, "treatment", "control")

# the outcome depends on the confounder only (no true group effect)
your_numerical_outcome <- 50 + 10 * (your_confounder == "high") + rnorm(N, 0, 8)

confound_numeric <- tibble(
  your_confounder           = factor(your_confounder, levels = c("low", "high")),
  your_explanatory_variable = factor(your_explanatory_variable, levels = c("control", "treatment")),
  your_numerical_outcome    = your_numerical_outcome
)

# Crude comparison ignores the confounder and can look like a group effect
confound_numeric |>
  group_by(your_explanatory_variable) |>
  summarize(n = n(), mean = mean(your_numerical_outcome), sd = sd(your_numerical_outcome))

# Within confounder levels the group gap largely disappears
confound_numeric |>
  group_by(your_confounder, your_explanatory_variable) |>
  summarize(n = n(), mean = mean(your_numerical_outcome), .groups = "drop")

### Case 4, Binary outcome, by chance (RCT, no true effect)

In [ ]:
set.seed(4)
n_per_group <- 80

# Random assignment, and BOTH groups share the same event probability:
# any difference in the "yes" rate is chance alone.
chance_binary <- tibble(
  your_explanatory_variable = factor(rep(c("control", "treatment"), each = n_per_group),
                                     levels = c("control", "treatment")),
  your_binary_outcome       = factor(c(rbinom(n_per_group, 1, 0.40),
                                       rbinom(n_per_group, 1, 0.40)),
                                     labels = c("no", "yes"))
)

chance_binary |>
  group_by(your_explanatory_variable) |>
  summarize(n = n(),
            yes      = sum(your_binary_outcome == "yes"),
            prop_yes = mean(your_binary_outcome == "yes"))

### Case 5, Binary outcome, causation (RCT, true effect)

In [ ]:
set.seed(5)
n_per_group <- 80

# Random assignment, plus a REAL effect: the treatment "yes" rate is truly higher.
cause_binary <- tibble(
  your_explanatory_variable = factor(rep(c("control", "treatment"), each = n_per_group),
                                     levels = c("control", "treatment")),
  your_binary_outcome       = factor(c(rbinom(n_per_group, 1, 0.40),
                                       rbinom(n_per_group, 1, 0.65)),
                                     labels = c("no", "yes"))
)

cause_binary |>
  group_by(your_explanatory_variable) |>
  summarize(n = n(),
            yes      = sum(your_binary_outcome == "yes"),
            prop_yes = mean(your_binary_outcome == "yes"))

### Case 6, Binary outcome, confounding (observational)

In [ ]:
set.seed(6)
N <- 300

# Observational: a confounder drives BOTH group membership AND the event,
# and the group itself has no true effect.
your_confounder <- sample(c("low", "high"), N, replace = TRUE, prob = c(0.5, 0.5))

# high-confounder units are more likely to be in the treatment group
p_treat <- ifelse(your_confounder == "high", 0.75, 0.25)
your_explanatory_variable <- ifelse(rbinom(N, 1, p_treat) == 1, "treatment", "control")

# the event probability depends on the confounder only (no true group effect)
p_event <- ifelse(your_confounder == "high", 0.60, 0.30)
your_binary_outcome <- ifelse(rbinom(N, 1, p_event) == 1, "yes", "no")

confound_binary <- tibble(
  your_confounder           = factor(your_confounder, levels = c("low", "high")),
  your_explanatory_variable = factor(your_explanatory_variable, levels = c("control", "treatment")),
  your_binary_outcome       = factor(your_binary_outcome, levels = c("no", "yes"))
)

# Crude comparison ignores the confounder
confound_binary |>
  group_by(your_explanatory_variable) |>
  summarize(n = n(), prop_yes = mean(your_binary_outcome == "yes"))

# Within confounder levels the group gap largely disappears
confound_binary |>
  group_by(your_confounder, your_explanatory_variable) |>
  summarize(n = n(), prop_yes = mean(your_binary_outcome == "yes"), .groups = "drop")

## Wrap-up

You simulated data under five designs, compared groups with descriptive statistics and plots, and saw six copy-and-adapt templates for the three explanations crossed with numerical and binary outcomes. Next, [Demo lab 2](module-4-lab-demo-tests) runs the matching tests and confidence intervals on the same kinds of data. For [Project 4](../projects/project-4) you will pick one template, rename the `your_*` variables, and carry that single two-group comparison through both demos.